<a href="https://colab.research.google.com/github/OmBoroleGit/AppliedGenAIAssignments/blob/main/04_structured_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- Colab setup: Runtime > Change runtime type > T4 GPU, then run this cell ------------------------
import os, sys
REPO = "AppliedGenAIAssignments"
if not os.path.exists(REPO) and not os.path.exists("src/model.py"):
    !git clone https://github.com/OmBoroleGit/AppliedGenAIAssignments.git
if os.path.exists(REPO):
    %cd {REPO}
sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())
assert os.path.exists("src/model.py"), "src/model.py missing - upload/push the project files first, then re-run"
!pip install -q -r requirements.txt

Cloning into 'AppliedGenAIAssignments'...
remote: Enumerating objects: 121, done.
remote: Counting objects: 100% (121/121), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 121 (delta 44), reused 90 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (121/121), 603.90 KiB | 17.25 MiB/s, done.
Resolving deltas: 100% (44/44), done.
/content/AppliedGenAIAssignments
working dir: /content/AppliedGenAIAssignments
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.7 MB/s eta 0:00:00


In [2]:
import json, re, os, pandas as pd
from collections import Counter
from src.model import load_model, generate_with_logprobs, set_seed, SEED
from src.schema import AnswerOutput, JSON_SCHEMA, validate_output, answer_json
from src.confidence import calibration_table
from src.runlog import load_eval_set, make_row, make_build_messages, run_cached, COLUMNS, EXTRA
set_seed(SEED)
build_msgs = make_build_messages("few_shot")   # team's best prompt from E2 (results/best_prompt.txt)
tok, model = load_model()
print("model on", model.device)
data = load_eval_set("data/eval_set.jsonl")
print(len(data), "questions;", sum(d["is_answerable"] for d in data), "answerable")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

model on cuda:0
200 questions; 100 answerable


In [3]:
print(json.dumps(JSON_SCHEMA, indent=2))

{
  "properties": {
    "answerable": {
      "title": "Answerable",
      "type": "boolean"
    },
    "answer": {
      "default": "",
      "title": "Answer",
      "type": "string"
    },
    "evidence": {
      "default": "",
      "title": "Evidence",
      "type": "string"
    },
    "confidence": {
      "anyOf": [
        {
          "maximum": 1.0,
          "minimum": 0.0,
          "type": "number"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Confidence"
    }
  },
  "required": [
    "answerable"
  ],
  "title": "AnswerOutput",
  "type": "object"
}


In [4]:
ctx = "Alexander Graham Bell patented the telephone in 1876. He was born in Edinburgh."
tests = {
 "good":              '{"answerable": true, "answer": "1876", "evidence": "Alexander Graham Bell patented the telephone in 1876.", "confidence": 0.8}',
 "good, 3 keys only": '{"answerable": true, "answer": "1876", "evidence": "Alexander Graham Bell patented the telephone in 1876."}',
 "fenced + chatter":  'Sure! ```json\n{"answerable": false, "answer": "", "evidence": ""}\n```',
 "no JSON":           "It was 1876.",
 "invented evidence": '{"answerable": true, "answer": "1877", "evidence": "Bell built it in 1877."}',
 "abstain + answer":  '{"answerable": false, "answer": "1876", "evidence": ""}',
 "team abstain":      '{"answerable": false, "answer": "not in the document", "evidence": ""}',
 "true but refuses":  '{"answerable": true, "answer": "not in the document", "evidence": "x"}',
 "wrong type":        '{"answerable": "yes", "answer": "", "evidence": ""}',
}
for name, raw in tests.items():
    r = validate_output(raw, ctx)
    print(f"{name:18s} ok={r.ok}  {r.errors}")

good               ok=True  []
good, 3 keys only  ok=True  []
fenced + chatter   ok=True  []
no JSON            ok=False  ['R1: no valid JSON object found in the reply']
invented evidence  ok=False  ['R5: evidence is not an exact quote from the paragraph']
abstain + answer   ok=False  ['R6: answerable is false but answer is not empty']
team abstain       ok=True  []
true but refuses   ok=False  ['R7: answerable is true but the answer says it is not in the document']
wrong type         ok=False  ["R2/R3: field 'answerable': Input should be a valid boolean"]


In [5]:
def run_e4(ex):
    out = answer_json(build_msgs(ex["context"], ex["question"]), ex["context"],
                      generate_with_logprobs, max_retries=1, temperature=0, seed=SEED)
    return make_row(ex, out, prompt_variant="few_shot", decoding="greedy")

df = run_cached("results/E4_structured_output.csv", data, run_e4, key=lambda ex: ex["id"])
df.drop(columns="_key").head()

  0%|          | 0/200 [00:00<?, ?it/s]

,id,question,context,gold_answers,is_answerable,prompt_variant,decoding,raw_output,pred_answerable,pred_answer,evidence,confidence,json_valid,attempts,json_valid_first_attempt,confidence_source,errors,warnings
0,57285ed5ff5b5019007da1b8,Who started the Yuan dynasty?,The Yuan dynasty (Chinese: 元朝; pinyin: Yuán Ch...,"[""Kublai Khan"", ""Kublai Khan"", ""Kublai Khan""]",True,few_shot,greedy,"{""answerable"": true, ""answer"": ""Kublai Khan"", ...",True,Kublai Khan,The Yuan dynasty (Chinese: 元朝; pinyin: Yuán Ch...,0.9673,True,1,True,token_probs,[],[]
1,56e16182e3433e1400422e29,By what main attribute are computational probl...,Computational complexity theory is a branch of...,"[""inherent difficulty"", ""their inherent diffic...",True,few_shot,greedy,"{""answerable"": true, ""answer"": ""inherent diffi...",True,inherent difficulty,Computational complexity theory is a branch of...,0.9278,True,1,True,token_probs,[],[]
2,572f5533a23a5019007fc55d,What is the largest city the Rhine runs through?,"The Rhine (Romansh: Rein, German: Rhein, Frenc...","[""Cologne, Germany"", ""Cologne, Germany"", ""Colo...",True,few_shot,greedy,"{""answerable"": true, ""answer"": ""Cologne"", ""evi...",True,Cologne,The biggest city on the river Rhine is Cologne...,0.9536,True,1,True,token_probs,[],[]
3,5a7b054521c2de001afe9cd0,How large is the San Joaquin Valley?,"Fresno (/ˈfrɛznoʊ/ FREZ-noh), the county seat ...",[],False,few_shot,greedy,"{""answerable"": false, ""answer"": """", ""evidence""...",False,,,0.5116,True,1,True,token_probs,[],[]
4,5a5504e1134fea001a0e17f9,What did the concept hope to achieve?,"Starting in the late 1950s, American computer ...",[],False,few_shot,greedy,"{""answerable"": true, ""answer"": ""fault-tolerant...",True,"fault-tolerant, efficient routing method","Starting in the late 1950s, American computer ...",0.8204,True,1,True,token_probs,[],[]


In [6]:
n = len(df)
first = df.json_valid_first_attempt.mean(); final = df.json_valid.mean()
print(f"JSON validity, first attempt : {first:.1%}")
print(f"JSON validity, after 1 retry : {final:.1%}   (rescued by retry: {int(round((final-first)*n))} of {n})")
errs = Counter(m for e in df.errors for m in re.findall(r"R\d", e))
print("failed-attempt reasons (R1 no JSON, R2/R3 types, R4 empty fields, R5 invented evidence, R6 abstain+answer, R7 true+refusal):")
print(dict(errs))
print("soft warnings:", Counter(m for w in df.warnings for m in re.findall(r"W\d", w)))
print("confidence source:", dict(df.confidence_source.value_counts()))

JSON validity, first attempt : 91.5%
JSON validity, after 1 retry : 100.0%   (rescued by retry: 17 of 200)
failed-attempt reasons (R1 no JSON, R2/R3 types, R4 empty fields, R5 invented evidence, R6 abstain+answer, R7 true+refusal):
{'R1': 4, 'R5': 11, 'R6': 2}
soft warnings: Counter({'W2': 10, 'W1': 5})
confidence source: {'token_probs': np.int64(200)}


In [7]:
show = pd.concat([df[df.is_answerable].head(4), df[~df.is_answerable].head(2)])
for _, r in show.iterrows():
    print("Q:", r.question, "| gold:", json.loads(r.gold_answers)[:1], "| answerable:", r.is_answerable)
    print("   ->", r.raw_output, "| valid:", r.json_valid, "| conf source:", r.confidence_source, "\n")

Q: Who started the Yuan dynasty? | gold: ['Kublai Khan'] | answerable: True
   -> {"answerable": true, "answer": "Kublai Khan", "evidence": "The Yuan dynasty (Chinese: 元朝; pinyin: Yuán Cháo), officially the Great Yuan (Chinese: 大元; pinyin: Dà Yuán; Mongolian: Yehe Yuan Ulus[a]), was the empire or ruling dynasty of China established by Kublai Khan, leader of the Mongolian Borjigin clan.", "confidence": 0.9673} | valid: True | conf source: token_probs 

Q: By what main attribute are computational problems classified utilizing computational complexity theory?  | gold: ['inherent difficulty'] | answerable: True
   -> {"answerable": true, "answer": "inherent difficulty", "evidence": "Computational complexity theory is a branch of the theory of computation in theoretical computer science that focuses on classifying computational problems according to their inherent difficulty, and relating those classes to each other.", "confidence": 0.9278} | valid: True | conf source: token_probs 

Q: What

In [8]:
def norm(s): return re.sub(r"[^a-z0-9 ]", "", re.sub(r"\s+", " ", str(s).lower())).strip()
def correct(row):
    gold = json.loads(row["gold_answers"])
    if not row["is_answerable"]: return not row["pred_answerable"]
    return bool(row["pred_answerable"]) and any(norm(g) == norm(row["pred_answer"]) or norm(g) in norm(row["pred_answer"]) for g in gold)
ok = df[df.json_valid].copy()
ok["correct"] = ok.apply(correct, axis=1)
table, ece = calibration_table(ok.confidence.tolist(), ok.correct.tolist(), n_bins=5)
print(pd.DataFrame(table)); print("Expected Calibration Error:", ece)
print("mean confidence when correct :", round(ok[ok.correct].confidence.mean(), 3))
print("mean confidence when wrong   :", round(ok[~ok.correct].confidence.mean(), 3))
print("loose accuracy:", round(ok.correct.mean(), 3))
pd.DataFrame(table).to_csv("results/summary_E4_confidence_calibration.csv", index=False)

       bin    n  mean_conf  accuracy
0  0.0-0.2    0        NaN       NaN
1  0.2-0.4    0        NaN       NaN
2  0.4-0.6   26      0.545     0.692
3  0.6-0.8   55      0.692     0.673
4  0.8-1.0  119      0.906     0.756
Expected Calibration Error: 0.1139
mean confidence when correct : 0.811
mean confidence when wrong   : 0.774
loose accuracy: 0.725


In [9]:
from datasets import load_dataset
from src import tools
print(json.dumps(tools.TOOL_SPEC, indent=2)[:1400])
sq = load_dataset("rajpurkar/squad_v2", split="validation")
# 20 of the 35 articles in OUR eval set, so demo questions are answerable from the index
titles = []
for d in data:
    t = d.get("title")
    if t and t not in titles: titles.append(t)
    if len(titles) == 20: break
corpus = tools.build_corpus(sq, titles=titles)
print(tools.init_index(corpus), "passages from", len(titles), "articles")
pool = [d for d in data if d.get("title") in set(titles)]
print(len(pool), "eval questions come from these 20 articles (", sum(d["is_answerable"] for d in pool), "answerable )")

{
  "name": "retrieve_context",
  "description": "Search a library of Wikipedia passages (SQuAD articles) with BM25 and return the passages most relevant to the question.",
  "input_schema": {
    "type": "object",
    "properties": {
      "question": {
        "type": "string",
        "description": "search query, normally the user question"
      },
      "top_k": {
        "type": "integer",
        "minimum": 1,
        "maximum": 5,
        "default": 3
      }
    },
    "required": [
      "question"
    ]
  },
  "output_schema": {
    "type": "object",
    "properties": {
      "passages": {
        "type": "array",
        "items": {
          "type": "object",
          "properties": {
            "passage_id": {
              "type": "integer"
            },
            "title": {
              "type": "string"
            },
            "text": {
              "type": "string"
            },
            "score": {
              "type": "number"
            }
          },


README.md:   0%|          | 0.00/8.92k [00:00<?, ?B/s]

squad_v2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 16.4MB            

squad_v2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

squad_v2/validation-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B / 1.35MB            

squad_v2/validation-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

676 passages from 20 articles
112 eval questions come from these 20 articles ( 56 answerable )


In [10]:
ans_pool = [d for d in pool if d["is_answerable"]]
hit1 = hit3 = 0
for d in ans_pool:
    texts = [p["text"] for p in tools.retrieve_context(d["question"], 3)["passages"]]
    hit1 += bool(texts) and texts[0] == d["context"]
    hit3 += d["context"] in texts
print(f"BM25 hit@1 = {hit1/len(ans_pool):.1%}   hit@3 = {hit3/len(ans_pool):.1%}   (n={len(ans_pool)})")

BM25 hit@1 = 62.5%   hit@3 = 76.8%   (n=56)


In [11]:
def run_e5(item):
    d, mode = item
    out = tools.answer_with_tools(d["question"], generate_with_logprobs,
                                  paragraph=None if mode == "no_paragraph" else d["context"],
                                  top_k=3, temperature=0, seed=SEED)
    r = make_row(d, out, prompt_variant=f"e5_tools_{mode}", decoding="greedy")
    r.update(mode=mode, tool_called=out["trace"]["tool_called"], forced=out["trace"]["forced_fallback"],
             n_passages=out["trace"]["n_passages"])
    return r

items = [(d, m) for d in pool for m in ("no_paragraph", "with_paragraph")]
d5 = run_cached("results/E5_tool_calling.csv", items, run_e5, key=lambda it: it[0]["id"] + "|" + it[1])
np_ = d5[d5["mode"] == "no_paragraph"]; wp = d5[d5["mode"] == "with_paragraph"]
print(f"no paragraph  : model called the tool itself {np_.tool_called.mean():.1%}, Python had to force the call {np_.forced.mean():.1%}")
print(f"with paragraph: model correctly skipped the tool {(~wp.tool_called).mean():.1%}")
for name, part in (("no_paragraph", np_), ("with_paragraph", wp)):
    v = part[part.json_valid].copy(); v["correct"] = v.apply(correct, axis=1)
    print(f"{name:15s} valid JSON {len(v)}/{len(part)}   loose accuracy {v.correct.mean():.1%}")

  0%|          | 0/224 [00:00<?, ?it/s]

no paragraph  : model called the tool itself 99.1%, Python had to force the call 0.9%
with paragraph: model correctly skipped the tool 38.4%
no_paragraph    valid JSON 112/112   loose accuracy 50.9%
with_paragraph  valid JSON 112/112   loose accuracy 58.9%


In [12]:
for d in [x for x in pool if x["is_answerable"]][:2] + [x for x in pool if not x["is_answerable"]][:1]:
    out = tools.answer_with_tools(d["question"], generate_with_logprobs, top_k=3, temperature=0, seed=SEED)
    print("QUESTION :", d["question"], "| gold:", d["gold_answers"][:1])
    print("TOOL ARGS:", out["trace"]["tool_args"], "| called by model:", out["trace"]["tool_called"])
    print("TOP PASSAGE:", (out["retrieved"][0]["title"] + ": " + out["retrieved"][0]["text"][:120] + "...") if out.get("retrieved") else None)
    print("FINAL    :", json.dumps(out["result"]), "\n")

QUESTION : Who started the Yuan dynasty? | gold: ['Kublai Khan']
TOOL ARGS: {'question': 'Who started the Yuan dynasty?', 'top_k': 3} | called by model: True
TOP PASSAGE: Yuan_dynasty: In 1271, Kublai Khan imposed the name Great Yuan (Chinese: 大元; pinyin: Dà Yuán; Wade–Giles: Ta-Yüan), establishing the Y...
FINAL    : {"answerable": true, "answer": "Kublai Khan", "evidence": "In 1271, Kublai Khan imposed the name Great Yuan (Chinese: \u5927\u5143; pinyin: D\u00e0 Yu\u00e1n; Wade\u2013Giles: Ta-Y\u00fcan), establishing the Yuan dynasty.", "confidence": 0.929} 

QUESTION : By what main attribute are computational problems classified utilizing computational complexity theory?  | gold: ['inherent difficulty']
TOOL ARGS: {'question': 'By what main attribute are computational problems classified utilizing computational complexity theory?', 'top_k': 3} | called by model: True
TOP PASSAGE: Computational_complexity_theory: Computational complexity theory is a branch of the theory of computation